# EXP-2026-010: primary test (does the image follow a labelled cavity?)

GPU runtime, after K1's training. Run all; send the SUMMARY block.

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"  # your fork only
BRANCH = "lumiere-onboarding"
WORK = "/content/MedSegFactory-k1-test"
DRIVE_RHUH = "/content/drive/MyDrive/RHUH-GBM"
SD_REPO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
MSF_REPO, MSF_FILE = "JohnWeck/StableDiffusion", "checkpoint-300.pth"
K1_TAG = "K1_r1"  # the RUN_TAG of K1's training run
RUN_TAG = "test_r1"  # a new tag for every attempt; a run folder that holds files is refused
DATA_ON_DRIVE = ("RHUH-GBM", "synthesis", "EXP-2026-001")
OUT_BASE, OUT_RUN = ("RHUH-GBM", "synthesis"), ("EXP-2026-010", "test", RUN_TAG)
LUMIERE_RUN = ("LUMIERE_derived", "EXP-2026-005", "r2")  # read only


In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # before any CUDA work: repeatable cuBLAS
from google.colab import drive

drive.mount("/content/drive")  # RHUH exports and processed files are read through the mount; writes go through the API
if not Path(WORK).is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
else:
    current = subprocess.check_output(["git", "-C", WORK, "branch", "--show-current"], text=True).strip()
    assert current == BRANCH, "This checkout belongs to another task; choose a different WORK"
subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
link = Path(WORK, "data", "RHUH-GBM")
link.parent.mkdir(exist_ok=True)
if not link.exists():
    link.symlink_to(DRIVE_RHUH, target_is_directory=True)
os.chdir(WORK)
# A session that ran another notebook may hold an older copy of the project in memory: use this checkout only
for key in [k for k in sys.modules if k.split(".")[0] in ("rhuh_synthesis", "rhuh_qc", "StableDiffusion")]:
    del sys.modules[key]
sys.path[:] = [p for p in sys.path if not p.startswith("/content/MedSegFactory")]
sys.path.insert(0, WORK)

from rhuh_synthesis import colab_packages  # the one package list (a module, so no name clash)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *colab_packages.DATA, *colab_packages.MODEL], check=True)
commit = subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(commit)

from google.colab import auth
from googleapiclient.discovery import build

from rhuh_synthesis.drive_api import DriveRun

auth.authenticate_user()  # Colab asks once for permission to use the Drive API with this account
api = build("drive", "v3", cache_discovery=False)
settings = json.loads(Path("configs/experiments/EXP-2026-010.json").read_text(encoding="utf-8"))
LOCAL_RUN = Path(f"/content/k1_{RUN_TAG}")
assert not LOCAL_RUN.exists(), f"{LOCAL_RUN} exists: choose a new RUN_TAG or a fresh runtime"
weights_runs = {
    "E0": DriveRun.open(api, DATA_ON_DRIVE, ("stage5_runs", "r2"), "/content/k1_test_w_E0", expect_in_base=("split", "export_v1")),
    "K1": DriveRun.open(api, OUT_BASE, ("EXP-2026-010", "train", K1_TAG), "/content/k1_test_w_K1", expect_in_base=("EXP-2026-001",)),
}
lumiere = DriveRun.open(api, LUMIERE_RUN[:-1], LUMIERE_RUN[-1:], "/content/k1_test_lumiere_r2", create=False)
run_drive = DriveRun.open(api, OUT_BASE, OUT_RUN, LOCAL_RUN, expect_in_base=("EXP-2026-001",), create=True)
access = run_drive.preflight()  # a real write, update, read and removal on Drive before any GPU time
print("free GiB on Drive:", None if access["free_gib"] is None else round(access["free_gib"], 1))
STARTED = time.perf_counter()


def say(message):
    print(f"[{time.perf_counter() - STARTED:6.0f} s] {message}", flush=True)


In [ ]:
import numpy as np
import torch
from diffusers import AutoencoderKL
from huggingface_hub import hf_hub_download
from transformers import CLIPTextModel, CLIPTokenizer

from rhuh_synthesis.cavity_targets import text_config_for
from rhuh_synthesis.model import reference_ddim_scheduler, scheduler_mismatches
from rhuh_synthesis.overfit import load_trained_unet
from rhuh_synthesis.provenance import code_version, config_sha256, environment
from rhuh_synthesis.stage3 import clip_encoder, load_reference_unet
from StableDiffusion.Our_UNet import UNet2DConditionModel

torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.benchmark = False
config = json.loads(Path("configs/experiments/EXP-2026-001.json").read_text(encoding="utf-8"))  # frozen, read only
d2 = json.loads(Path("configs/experiments/EXP-2026-008_d2.json").read_text(encoding="utf-8"))  # frozen, read only
text_config = text_config_for(config["text"], settings["text_class_names"])  # the same prompts for both models
sources, weight_files = {}, {}
for name, run in weights_runs.items():
    sources[name] = json.loads(run.fetch("record.json").read_text(encoding="utf-8"))  # by ID, checked by MD5
    weight_files[name] = run.fetch(sources[name]["weights"]["file"])
assert sources["K1"]["run"] == "K1" and sources["K1"]["weights"]["step"] == settings["run"]["evaluated_checkpoint_step"], \
    "K1's training record belongs to another run or step: stop and report"
assert sources["K1"]["gate"]["pass"], "K1's training record does not hold a passed gate: stop and report"
checkpoint = hf_hub_download(MSF_REPO, MSF_FILE)
vae = AutoencoderKL.from_pretrained(SD_REPO, subfolder="vae").to("cuda").eval().requires_grad_(False)
tokenizer = CLIPTokenizer.from_pretrained(SD_REPO, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(SD_REPO, subfolder="text_encoder").to("cuda").eval().requires_grad_(False)
ddim = reference_ddim_scheduler()
assert scheduler_mismatches(ddim) == []
encode_text = clip_encoder(tokenizer, text_encoder, "cuda")


def load_model(name):
    unet4, _ = load_reference_unet(UNet2DConditionModel, UNet2DConditionModel.load_config(SD_REPO, subfolder="unet"),
                                   checkpoint)
    assert unet4 is not None, "Checkpoint keys do not match the vendored UNet: stop and report"
    unet, report = load_trained_unet(unet4, weight_files[name], device="cuda")
    assert report["sha256"] == sources[name]["weights"]["sha256"], f"{name}: weights differ from their record"
    return unet, report


record = {
    "experiment": "EXP-2026-010", "part": "primary test", "run_tag": RUN_TAG, "k1_tag": K1_TAG, "commit": commit,
    "code_version": code_version(WORK), "settings_sha256": config_sha256(settings),
    "exp001_config_sha256": config_sha256(config), "d2_config_sha256": config_sha256(d2), "environment": environment(),
    "torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
    "text_config": text_config, "test": settings["test"],
}
say("encoders ready")


In [ ]:
from rhuh_synthesis.e_series import accepted_development_pairs, heldout_labels, rhuh_training_members
from rhuh_synthesis.lumiere_scaffold_check import verified_split
from rhuh_synthesis.pilot import accepted_patients, local_copies

PRIVATE = Path("data/RHUH-GBM/synthesis/EXP-2026-001")
split = json.loads((PRIVATE / "split/split.json").read_text(encoding="utf-8"))
policy = json.loads((PRIVATE / "registration_policy/acceptance_policy_v1.json").read_text(encoding="utf-8"))
members = rhuh_training_members(split, policy)
val_dir = PRIVATE / "export_v1/val"
val_patients = accepted_patients(split["roles"]["val"]["patients"], val_dir)
assert not set(val_patients) & set(members) and not set(val_patients) & set(split["roles"]["test"]["patients"])
for name in ("run.json", "split.json", "registration_qc.csv", "summary.json"):
    lumiere.fetch(f"outputs/{name}")
outputs = Path("/content/k1_test_lumiere_r2/outputs")
lumiere_split, lumiere_split_sha, _ = verified_split(outputs)
lumiere.fetch_folder("outputs/export/development")  # generated from pre-op inputs only; no post-op value is read
dev_pairs, dev_counts = accepted_development_pairs(outputs, lumiere_split, lumiere_split_sha)
expected = settings["test"]["expected_heldout"]
assert len(val_patients) == expected["RHUH"] and dev_counts["accepted"] == expected["LUMIERE"], "Held-out counts differ from the card"
LABELS = heldout_labels(val_patients, [path.stem for path, _ in dev_pairs])
rhuh_paths = local_copies({p: val_dir / f"{p}.npz" for p in val_patients}, "/content/k1_test_val")
dev_paths = {path.stem: path for path, _ in dev_pairs}


def heldout_arrays(label):
    cohort, item = LABELS[label]
    with np.load(rhuh_paths[item] if cohort == "RHUH" else dev_paths[item]) as data:
        return cohort, {key: data[key] for key in data.files}


say(f"held out: {expected}")


In [ ]:
from rhuh_synthesis.cavity_test import run_patient, test_inputs

tc = settings["test"]
results = {}
torch.cuda.reset_peak_memory_stats()
for name in ("E0", "K1"):
    unet, report = load_model(name)
    MODELS = {"unet": unet, "vae": vae, "scheduler": ddim, "encode_text": encode_text,
              "codec": config["representation"]["mask_codec"], "crop_mm": config["representation"]["crop_mm"],
              "out_px": config["representation"]["out_px"], "num_inference_steps": config["sampling"]["num_inference_steps"],
              "guidance_scale": config["sampling"]["guidance_scale"], "device": "cuda"}
    results[name] = {"weights": report, "patients": {}}
    for rank, label in enumerate(sorted(LABELS)):
        cohort, arrays = heldout_arrays(label)
        inputs = test_inputs(arrays, tc, d2, rank)
        seeds = run_patient(arrays, inputs, models=MODELS, text_config=text_config, seeds=tc["seeds"],
                            starts=d2["starts"], batch=tc["batch_pairs"], progress=lambda m: say(f"{name} {label} {m}"))
        results[name]["patients"][label] = {"cohort": cohort, "seeds": seeds, "level": inputs["level"],
                                            "slab_start": inputs["slab_start"]}
    del unet, MODELS
    torch.cuda.empty_cache()
record["results"] = results
record["peak_gib"] = torch.cuda.max_memory_allocated() / 2**30


In [ ]:
from rhuh_synthesis.cavity_test import responses, test_decision
from rhuh_synthesis.manifest import write_once_json

decision = test_decision(results["K1"], results["E0"], settings["test"]["bootstrap"])
record.update(decision=decision, seconds=time.perf_counter() - STARTED, items={l: list(v) for l, v in LABELS.items()})
LOCAL_RUN.mkdir(parents=True, exist_ok=True)
path, digest = write_once_json(LOCAL_RUN / "k1_test_record.json", record)
report = run_drive.sync()
say(f"record {digest[:12]} written; Drive upload: {report}")
shareable = {"commit": commit, "gpu": record["gpu"], "seconds": round(record["seconds"]), "peak_gib": round(record["peak_gib"], 1),
             "per_patient": {name: {label: {"cohort": c, "B_minus_A_mean": round(m, 4), "seed_range": round(r, 4),
                                             "start": round(results[name]["patients"][label]["seeds"][str(tc["seeds"][0])]["start"], 4)}
                                     for label, (c, m, r) in responses(results[name]).items()} for name in ("E0", "K1")},
             "decision": decision}


In [ ]:
assert run_drive.confirm(), f"Not every file is confirmed on Drive. The originals are in {LOCAL_RUN}: keep this session open and report"
print("Every file confirmed by the Drive server: MyDrive/" + "/".join(OUT_BASE + OUT_RUN))
print("SUMMARY")
print(json.dumps(shareable, indent=1))
